# Data Preparation — NautiCost

This notebook cleans and structures the raw NautiCost data files and exports ready-to-use CSVs.

**Input files (raw):**
- `Kostnader_MM.csv` — transaction-level cost/charge data (all yachts, 2020–2025)
- `Yacht-specs.csv` — physical and operational specifications per yacht
- `cockpit_2020.csv` … `cockpit_2025.csv` — aggregated KPIs per year

**Output files (clean):**
- `costs_clean.csv` — parsed, typed, flagged transaction data with standardised service categories
- `specs_clean.csv` — validated and typed yacht specifications
- `cockpit_clean.csv` — one row per year with revenue, gross margin, and network KPIs
- `costs_merged.csv` — transactions joined with yacht specs (analysis-ready)

In [1]:
import pandas as pd
import numpy as np
import re
import csv
from pathlib import Path

DATA_DIR = Path('../004 data')
print('Data directory:', DATA_DIR.resolve())

Data directory: C:\Users\jorge\OneDrive - Høgskolen i Molde\LOG650 - Forskningsprosjekt\LOG650_NautiCost\G11-jorgen-individuell\004 data


---
## 1. Kostnader_MM.csv — Transaction Data

### 1.1 Known Raw Data Issues

| Issue | Location | Fix |
|-------|----------|-----|
| `Yacht:_2` (colon typo) | ~line 2637 | Replace with `Yacht_2` |
| `Yacht:_14` (colon typo) | ~line 3222 | Replace with `Yacht_14` |
| Subtotal rows (no dates, only amounts) | After each transaction group | Skip — do not parse as transactions |
| `Not set` in VAT columns | Systematic | Treat as NaN — VAT not tracked |
| Two column layouts (standard vs extended with `viewedit` prefix) | Sections vary | Detect by col[8] content |

In [2]:
def looks_like_money(s: str) -> bool:
    """True if string represents a monetary value (e.g. '3,665.83' or '0.00')."""
    s = s.strip()
    return bool(re.match(r'^[\d,]+\.\d{2}$', s))

def parse_money(s) -> float:
    """Parse '3,665.83' or '0.00' to float; returns NaN for missing/non-numeric."""
    if pd.isna(s):
        return np.nan
    s = str(s).strip()
    if s in ('', 'Not set'):
        return np.nan
    try:
        return float(s.replace(',', ''))
    except ValueError:
        return np.nan

date_pattern = re.compile(r'\d{4}-\d{2}-\d{2}')
yacht_id_pattern = re.compile(r'^Yacht[_:]\d+$', re.IGNORECASE)

records = []
current_yacht = None
issues_found = []

with open(DATA_DIR / 'Kostnader_MM.csv', 'r', encoding='utf-8') as f:
    reader = csv.reader(f)
    for lineno, row in enumerate(reader, start=1):
        if not row or len(row) < 4:
            continue

        first = row[0].strip()

        # --- Detect and fix yacht ID header rows ---
        if yacht_id_pattern.match(first):
            raw_id = first
            # Fix colon typo: Yacht:_2 → Yacht_2
            fixed_id = re.sub(r'Yacht[_:](\d+)', lambda m: f'yacht_{m.group(1)}', raw_id, flags=re.IGNORECASE)
            if raw_id != fixed_id.upper().replace('YACHT_', 'Yacht_'):
                issues_found.append(f'Line {lineno}: Fixed yacht ID "{raw_id}" → "{fixed_id}"')
            current_yacht = fixed_id.lower()
            continue

        # Skip column header rows
        if first in ('#', 'Clear') or 'Service Type' in str(row[1]):
            continue
        if current_yacht is None:
            continue

        # Must have a date in col 3 to be a transaction row
        if not (len(row) > 3 and date_pattern.match(str(row[3]).strip())):
            continue

        # Detect format: standard (col[0] empty) vs extended (col[0] = 'viewedit')
        col8 = row[8].strip() if len(row) > 8 else ''
        is_standard = looks_like_money(col8) or col8 in ('0.00', '')

        try:
            if is_standard:
                office          = row[1].strip()
                arrival_port    = row[2].strip()
                arrival_date    = row[3].strip()
                departure_date  = row[4].strip()
                service_type    = row[5].strip()
                invoice_comments= row[6].strip()
                supplier        = row[7].strip()
                cost_no_vat     = row[8].strip()
                final_charge    = row[15].strip() if len(row) > 15 else ''
            else:
                # Extended format: col[0] = 'viewedit', one extra column shifts everything
                office          = row[1].strip()
                arrival_port    = row[2].strip()
                arrival_date    = row[3].strip()
                departure_date  = row[4].strip()
                service_type    = row[5].strip()
                invoice_comments= row[7].strip()
                supplier        = row[8].strip()
                cost_no_vat     = row[9].strip()
                final_charge    = row[16].strip() if len(row) > 16 else ''

            records.append({
                'yacht_id'        : current_yacht,
                'office'          : office,
                'arrival_port'    : arrival_port,
                'arrival_date'    : arrival_date,
                'departure_date'  : departure_date,
                'service_type'    : service_type,
                'invoice_comments': invoice_comments,
                'supplier'        : supplier,
                'cost_no_vat'     : cost_no_vat,
                'final_charge'    : final_charge,
                'row_format'      : 'standard' if is_standard else 'extended',
            })
        except IndexError:
            issues_found.append(f'Line {lineno}: Too few columns ({len(row)}), skipped')

costs_raw = pd.DataFrame(records)
print(f'Parsed rows: {len(costs_raw)}')
print(f'\nIssues fixed during parsing:')
for issue in issues_found:
    print(' ', issue)

Parsed rows: 1654

Issues fixed during parsing:


### 1.2 Type Conversion and Derived Columns

In [3]:
costs = costs_raw.copy()

# Numeric columns
costs['cost_no_vat']  = costs['cost_no_vat'].apply(parse_money)
costs['final_charge'] = costs['final_charge'].apply(parse_money)

# Dates
costs['arrival_date']   = pd.to_datetime(costs['arrival_date'],   errors='coerce')
costs['departure_date'] = pd.to_datetime(costs['departure_date'], errors='coerce')

# Derived columns
costs['year']      = costs['arrival_date'].dt.year
costs['month']     = costs['arrival_date'].dt.month
costs['stay_days'] = (costs['departure_date'] - costs['arrival_date']).dt.days
costs['margin']    = costs['final_charge'] - costs['cost_no_vat']

print('Datatypes:')
print(costs.dtypes)
print(f'\nYear distribution:')
print(costs['year'].value_counts().sort_index())

Datatypes:
yacht_id                       str
office                         str
arrival_port                   str
arrival_date        datetime64[us]
departure_date      datetime64[us]
service_type                   str
invoice_comments               str
supplier                       str
cost_no_vat                float64
final_charge               float64
row_format                     str
year                         int32
month                        int32
stay_days                  float64
margin                     float64
dtype: object

Year distribution:
year
2020     26
2021    137
2022    129
2023    195
2024    490
2025    670
2026      7
Name: count, dtype: int64


### 1.3 Service Type Mapping

The raw data contains 58 granular service types. We map these to the 7 standard cockpit
categories so that transaction-level data can be compared with the aggregated cockpit KPIs.

| Cockpit Category | Description |
|-----------------|-------------|
| Agency Fee | Agency fees, admin fees, agent time |
| Agency Services | Clearance, courier, logistics, purchasing, storage, delivery, crew, misc |
| Bunkering | Fuel, lube oil |
| Hospitality | Transfers, car rental, tours, hotel, laundry, airport |
| Port Marina | Port dues, pilotage, mooring, NOx tax, marina fees, garbage, water removal |
| Provisioning | Provisions, provisioning assistance, florist |
| Technical Services | Technical, mechanical, hydraulic, diver, carpenter, repair, dry dock, boat work, internet |

In [4]:
SERVICE_TYPE_MAP = {
    # Agency Fee
    'Agency Fees':              'Agency Fee',
    'Administrative Fees':      'Agency Fee',
    'Agent Time & Service':     'Agency Fee',

    # Agency Services
    'Agency Services':          'Agency Services',
    'Custom Clearance':         'Agency Services',
    'Custom Formalities':       'Agency Services',
    'Clearance in/out':         'Agency Services',
    'Immigration Formalities':  'Agency Services',
    'Purchasing Assistance':    'Agency Services',
    'Storage Arrangements':     'Agency Services',
    'Storage & Transport':      'Agency Services',
    'Courier':                  'Agency Services',
    'Delivery Charge':          'Agency Services',
    'Delivery Services':        'Agency Services',
    'Medical Arrangements':     'Agency Services',
    'Doctor/Medical':           'Agency Services',
    'Logistic Arrangements':    'Agency Services',
    'International Transport':  'Agency Services',
    'Speciality Shipment Items':'Agency Services',
    'Parcel Transfer':          'Agency Services',
    'Service':                  'Agency Services',
    'Crew Placement':           'Agency Services',

    # Bunkering
    'Fuel - Diesel':            'Bunkering',
    'Lube Oil Supply':          'Bunkering',

    # Hospitality
    'Transfer Service':         'Hospitality',
    'Car Rental':               'Hospitality',
    'Rental Car':               'Hospitality',
    'Guide Services':           'Hospitality',
    'Tours / Excursions':       'Hospitality',
    'Tourism or Guide / Tour':  'Hospitality',
    'Transportation':           'Hospitality',
    'Airport Transfer':         'Hospitality',
    'Hotel Accommodation':      'Hospitality',
    'Laundry/Dry Cleaning':     'Hospitality',
    'Dry Cleaning Services':    'Hospitality',

    # Port Marina
    'Port Dues':                'Port Marina',
    'Arrival Pilot Fees':       'Port Marina',
    'Pilot Fees Arrival':       'Port Marina',
    'Mooring/Unmooring Assistance': 'Port Marina',
    'NOx Tax':                  'Port Marina',
    'Marina Water Fees':        'Port Marina',
    'Garbage Disposal':         'Port Marina',
    'Sludge/Oil/Water Removal': 'Port Marina',
    'Dirty Water Removal Service': 'Port Marina',

    # Provisioning
    'Provisioning':             'Provisioning',
    'Provisioning Assistance':  'Provisioning',
    'Florist Services':         'Provisioning',

    # Technical Services
    'Technical Assistance':     'Technical Services',
    'Technician':               'Technical Services',
    'Carpenter Service':        'Technical Services',
    'Mechanical Services':      'Technical Services',
    'Hydraulic Services':       'Technical Services',
    'Diver Services':           'Technical Services',
    'Repair of':                'Technical Services',
    'Dry Dock/Shipyard Services':'Technical Services',
    'Boat Work':                'Technical Services',
    'Internet Connection':      'Technical Services',
}

costs['service_category'] = costs['service_type'].map(SERVICE_TYPE_MAP)

# Check for unmapped service types
unmapped = costs[costs['service_category'].isna()]['service_type'].unique()
if len(unmapped) > 0:
    print(f'WARNING — unmapped service types: {unmapped}')
else:
    print('All 58 service types mapped to 7 cockpit categories.')

print('\nService category distribution:')
print(costs['service_category'].value_counts())

All 58 service types mapped to 7 cockpit categories.

Service category distribution:
service_category
Agency Services       599
Port Marina           387
Hospitality           271
Provisioning          150
Technical Services    131
Agency Fee             75
Bunkering              41
Name: count, dtype: int64


### 1.4 Validation and Flagging

In [5]:
print('=== MISSING VALUES ===')
print(costs.isnull().sum())

print('\n=== NEGATIVE STAY DAYS ===')
neg_stay = costs[costs['stay_days'] < 0]
print(f'{len(neg_stay)} rows with negative stay (departure before arrival)')
if len(neg_stay) > 0:
    print(neg_stay[['yacht_id', 'arrival_date', 'departure_date', 'service_type']].head(10))

print('\n=== NEGATIVE/ZERO FINAL CHARGE ===')
zero_charge = costs[costs['final_charge'] <= 0]
print(f'{len(zero_charge)} rows with final_charge <= 0')
if len(zero_charge) > 0:
    print(zero_charge[['yacht_id', 'year', 'service_type', 'cost_no_vat', 'final_charge']].head(10))

print('\n=== UNKNOWN YACHT IDs ===')
known_yachts = {f'yacht_{i}' for i in range(1, 18)}
unknown = costs[~costs['yacht_id'].isin(known_yachts)]['yacht_id'].unique()
print(f'Unknown yacht IDs: {unknown}')

print('\n=== ROW FORMAT DISTRIBUTION ===')
print(costs['row_format'].value_counts())

print('\n=== YEAR RANGE CHECK ===')
out_of_range = costs[~costs['year'].between(2020, 2025)]
print(f'{len(out_of_range)} rows outside 2020–2025')
if len(out_of_range) > 0:
    print(out_of_range[['yacht_id', 'arrival_date', 'service_type']].head(10))

=== MISSING VALUES ===
yacht_id             0
office               0
arrival_port         0
arrival_date         0
departure_date      25
service_type         0
invoice_comments     0
supplier             0
cost_no_vat         19
final_charge         2
row_format           0
year                 0
month                0
stay_days           25
margin              19
service_category     0
dtype: int64

=== NEGATIVE STAY DAYS ===
0 rows with negative stay (departure before arrival)

=== NEGATIVE/ZERO FINAL CHARGE ===
19 rows with final_charge <= 0
      yacht_id  year                  service_type  cost_no_vat  final_charge
1353   yacht_6  2025                     Port Dues          0.0           0.0
1358   yacht_6  2025  Mooring/Unmooring Assistance          0.0           0.0
1530  yacht_12  2025                   Agency Fees          NaN           0.0
1535  yacht_13  2025                   Agency Fees          NaN           0.0
1536  yacht_13  2025              Clearance in/out        

In [6]:
# Flag rows with data quality concerns (do not remove — flag for transparency)
costs['flag'] = ''
costs.loc[costs['final_charge'].isna(), 'flag'] += 'missing_charge;'
costs.loc[costs['cost_no_vat'].isna(), 'flag'] += 'missing_cost;'
costs.loc[costs['arrival_date'].isna(), 'flag'] += 'missing_date;'
costs.loc[(costs['stay_days'].notna()) & (costs['stay_days'] < 0), 'flag'] += 'negative_stay;'
costs.loc[(costs['final_charge'].notna()) & (costs['final_charge'] == 0), 'flag'] += 'zero_charge;'
costs.loc[~costs['year'].between(2020, 2025), 'flag'] += 'out_of_year_range;'

n_flagged = (costs['flag'] != '').sum()
print(f'Flagged rows: {n_flagged} / {len(costs)}')
if n_flagged > 0:
    print(costs[costs['flag'] != ''][['yacht_id', 'year', 'service_type', 'final_charge', 'flag']].head(20))

Flagged rows: 28 / 1654
      yacht_id  year                  service_type  final_charge  \
1046   yacht_2  2026           Storage & Transport      16438.75   
1047   yacht_2  2026           Storage & Transport     105000.00   
1048   yacht_2  2026         Purchasing Assistance      23450.00   
1054   yacht_2  2026           Storage & Transport      17063.75   
1055   yacht_2  2026          Storage Arrangements     116250.00   
1061   yacht_2  2026          Storage Arrangements      18770.13   
1062   yacht_2  2026          Storage Arrangements     116250.00   
1270   yacht_4  2025                     Port Dues           NaN   
1353   yacht_6  2025                     Port Dues          0.00   
1358   yacht_6  2025  Mooring/Unmooring Assistance          0.00   
1491  yacht_11  2025         Purchasing Assistance           NaN   
1530  yacht_12  2025                   Agency Fees          0.00   
1535  yacht_13  2025                   Agency Fees          0.00   
1536  yacht_13  2025    

---
## 2. Yacht-specs.csv — Specifications

### 2.1 Known Issues

| Yacht | Issue | Action |
|-------|-------|--------|
| yacht_17 | GT = 0 | Flag as suspect; set to NaN |
| yacht_1 | Air Draft = 0 | Flag as suspect; set to NaN |
| yacht_7 | NT missing | Keep as NaN |
| yacht_9, yacht_13, yacht_16, yacht_17 | Air Draft missing | Keep as NaN |
| yacht_2-1/2-2/2-3 | Three variants for one cost ID | Keep variants; create `yacht_id_base = yacht_2` for joining |

In [7]:
specs = pd.read_csv(DATA_DIR / 'Yacht-specs.csv')
specs = specs.dropna(how='all').reset_index(drop=True)
specs = specs.loc[:, ~specs.columns.str.startswith('Unnamed')].copy()

# Normalize yacht ID
specs['yacht_id']      = specs['Yacht'].str.strip().str.lower()
specs['yacht_id_base'] = specs['yacht_id'].str.replace(r'-\d+$', '', regex=True)

# Numeric columns
num_cols = ['GT', 'NT', 'LOA (m)', 'Reg. Length (m)', 'Beam (m)', 'Draft (m)',
            'Air Draft (m)', 'Fuel consumption (L/h)']
for col in num_cols:
    specs[col] = pd.to_numeric(specs[col], errors='coerce')

print('Loaded specs:')
specs[['yacht_id', 'yacht_id_base', 'GT', 'NT', 'LOA (m)', 'Air Draft (m)',
       'Fuel consumption (L/h)', 'Loskrav (LOA>70m)', 'Størrelses-kategori']]

Loaded specs:


,yacht_id,yacht_id_base,GT,NT,LOA (m),Air Draft (m),Fuel consumption (L/h),Loskrav (LOA>70m),Størrelses-kategori
0,yacht_1,yacht_1,52.92,52.92,23.98,0.0,30,Nei,Liten
1,yacht_2-1,yacht_2,51.90,51.90,18.90,32.0,25,Nei,Liten
2,yacht_2-2,yacht_2,51.90,51.90,18.90,32.0,25,Nei,Liten
3,yacht_2-3,yacht_2,51.90,51.90,18.90,32.0,25,Nei,Liten
4,yacht_3,yacht_3,206.00,61.00,36.50,42.0,80,Nei,Mellomstor
5,yacht_4,yacht_4,2407.00,722.00,77.40,25.0,500,Ja,Stor
6,yacht_5,yacht_5,98.00,29.00,33.00,40.0,50,Nei,Mellomstor
7,yacht_6,yacht_6,2149.00,644.00,79.20,26.0,450,Ja,Stor
8,yacht_7,yacht_7,161.84,NaN,28.58,14.0,60,Nei,Mellomstor
9,yacht_8,yacht_8,658.00,197.00,50.00,16.5,150,Nei,Mellomstor


In [8]:
specs_clean = specs.copy()

# GT = 0 on yacht_17 is almost certainly a data entry error
mask_gt0 = (specs_clean['GT'] == 0)
if mask_gt0.any():
    print(f'GT = 0 (set to NaN): {specs_clean.loc[mask_gt0, "yacht_id"].tolist()}')
    specs_clean.loc[mask_gt0, 'GT'] = np.nan

# Air Draft = 0 on yacht_1 — no superyacht has zero air draft; likely missing
mask_ad0 = (specs_clean['Air Draft (m)'] == 0)
if mask_ad0.any():
    print(f'Air Draft = 0 (set to NaN): {specs_clean.loc[mask_ad0, "yacht_id"].tolist()}')
    specs_clean.loc[mask_ad0, 'Air Draft (m)'] = np.nan

# Categorical — standardize casing
specs_clean['Størrelses-kategori'] = specs_clean['Størrelses-kategori'].str.strip().str.capitalize()
specs_clean['Loskrav (LOA>70m)']   = specs_clean['Loskrav (LOA>70m)'].str.strip().str.capitalize()

print('\nMissing values after cleaning:')
print(specs_clean[num_cols + ['Loskrav (LOA>70m)', 'Størrelses-kategori']].isnull().sum())

GT = 0 (set to NaN): ['yacht_17']
Air Draft = 0 (set to NaN): ['yacht_1']

Missing values after cleaning:
GT                        1
NT                        2
LOA (m)                   0
Reg. Length (m)           0
Beam (m)                  0
Draft (m)                 0
Air Draft (m)             5
Fuel consumption (L/h)    0
Loskrav (LOA>70m)         0
Størrelses-kategori       0
dtype: int64


In [9]:
# Sanity checks
print('=== GT vs LOA CONSISTENCY ===')
print(specs_clean[['yacht_id', 'GT', 'LOA (m)', 'Størrelses-kategori']].sort_values('LOA (m)').to_string(index=False))

print('\n=== LOSKRAV CHECK (should be Ja for LOA > 70m) ===')
loskrav_check = specs_clean[['yacht_id', 'LOA (m)', 'Loskrav (LOA>70m)']].copy()
loskrav_check['expected_loskrav'] = loskrav_check['LOA (m)'].apply(
    lambda x: 'Ja' if pd.notna(x) and x > 70 else 'Nei'
)
mismatch = loskrav_check[loskrav_check['Loskrav (LOA>70m)'] != loskrav_check['expected_loskrav']]
if len(mismatch) > 0:
    print('Mismatches found:')
    print(mismatch)
else:
    print('All loskrav values consistent with LOA.')

print('\n=== FUEL CONSUMPTION COVERAGE ===')
no_fuel = specs_clean[specs_clean['Fuel consumption (L/h)'].isna()]['yacht_id'].tolist()
print(f'Yachts missing fuel consumption: {no_fuel}')

=== GT vs LOA CONSISTENCY ===
 yacht_id      GT  LOA (m) Størrelses-kategori
yacht_2-1   51.90    18.90               Liten
yacht_2-2   51.90    18.90               Liten
yacht_2-3   51.90    18.90               Liten
  yacht_9   95.00    23.96               Liten
  yacht_1   52.92    23.98               Liten
 yacht_17     NaN    24.85               Liten
  yacht_7  161.84    28.58          Mellomstor
 yacht_14   99.00    30.02               Liten
 yacht_15  138.00    31.59          Mellomstor
  yacht_5   98.00    33.00          Mellomstor
 yacht_12  143.00    34.99          Mellomstor
  yacht_3  206.00    36.50          Mellomstor
 yacht_13  496.00    46.96          Mellomstor
  yacht_8  658.00    50.00          Mellomstor
 yacht_10  497.00    53.25          Mellomstor
 yacht_11 1143.00    60.20                Stor
 yacht_16 1161.00    62.50                Stor
  yacht_4 2407.00    77.40                Stor
  yacht_6 2149.00    79.20                Stor

=== LOSKRAV CHECK (should be 

### 2.2 Create join-ready specs (one row per yacht_id_base)

For yacht_2 we have three hull variants (2-1, 2-2, 2-3) but only one cost ID (`yacht_2`).
We aggregate specs by taking the mean of numeric columns per `yacht_id_base` so we get
a single row that can join cleanly with the cost data.

In [10]:
# Aggregate yacht_2 variants into one row per yacht_id_base
specs_join = (
    specs_clean
    .groupby('yacht_id_base')
    .agg({
        **{col: 'mean' for col in num_cols},
        'Loskrav (LOA>70m)': 'first',
        'Størrelses-kategori': 'first',
    })
    .reset_index()
    .rename(columns={'yacht_id_base': 'yacht_id'})
)

# Rename columns to snake_case for easier downstream use
specs_join = specs_join.rename(columns={
    'GT': 'gt',
    'NT': 'nt',
    'LOA (m)': 'loa_m',
    'Reg. Length (m)': 'reg_length_m',
    'Beam (m)': 'beam_m',
    'Draft (m)': 'draft_m',
    'Air Draft (m)': 'air_draft_m',
    'Fuel consumption (L/h)': 'fuel_lph',
    'Loskrav (LOA>70m)': 'loskrav',
    'Størrelses-kategori': 'size_category',
})

print(f'Specs join table: {len(specs_join)} yachts')
specs_join

Specs join table: 17 yachts


,yacht_id,gt,nt,loa_m,reg_length_m,beam_m,draft_m,air_draft_m,fuel_lph,loskrav,size_category
0,yacht_1,52.92,52.92,23.98,21.59,5.95,2.500000,NaN,30.0,Nei,Liten
1,yacht_10,497.00,149.00,53.25,50.88,9.20,3.050000,17.0,130.0,Nei,Mellomstor
2,yacht_11,1143.00,342.00,60.20,56.80,11.00,3.700000,19.7,250.0,Nei,Stor
3,yacht_12,143.00,43.00,34.99,33.47,8.00,4.000000,49.0,55.0,Nei,Mellomstor
4,yacht_13,496.00,148.00,46.96,41.62,9.27,3.560000,NaN,120.0,Nei,Mellomstor
5,yacht_14,99.00,34.00,30.02,30.02,6.70,4.000000,44.0,45.0,Nei,Liten
6,yacht_15,138.00,41.00,31.59,28.91,6.02,3.000000,15.0,55.0,Nei,Mellomstor
7,yacht_16,1161.00,348.00,62.50,56.03,10.30,3.500000,NaN,260.0,Nei,Stor
8,yacht_17,NaN,NaN,24.85,22.14,6.22,2.840000,NaN,20.0,Nei,Liten
9,yacht_2,51.90,51.90,18.90,18.90,10.00,2.436667,32.0,25.0,Nei,Liten


---
## 3. Cockpit CSVs (2020–2025) — Aggregated KPIs

### 3.1 Known Issues

| Issue | Detail | Action |
|-------|--------|--------|
| Avg. days per call/boat | Stored as Excel serial dates in CSV → unreadable | Skip these columns |
| 2020 format shift | Labels in col[0] instead of col[1] | Detected automatically |
| Empty/zero values for early years | 2020 had minimal activity | Kept as NaN / 0 |
| `revenue_Total` for 2024/2025 | Parsed as 1181699 / 1541399 instead of ~1181.699 | Fix decimal point |

In [11]:
COCKPIT_SERVICE_TYPES = {'Agency Fee', 'Agency Services', 'Bunkering', 'Hospitality',
                         'Port Marina', 'Provisioning', 'Technical Services', 'Total'}

SECTION_HEADERS = {
    'Revenues':        'revenue',
    'Gross Margin':    'gm',
    'Gross Margin (%)':'gm_pct',
}

def _to_float(s: str):
    """Convert string to float; return NaN for empty/non-numeric/date-like values."""
    s = str(s).strip()
    if not s or s in ('N/A', 'Not set'):
        return np.nan
    # Skip values that look like dates (Excel serial date artefacts)
    if re.match(r'\d{4}-\d{2}-\d{2}', s):
        return np.nan
    try:
        return float(s.replace(',', ''))
    except ValueError:
        return np.nan

def parse_cockpit_file(filepath: Path, year: int) -> dict:
    """
    Parse one cockpit CSV into a flat dict of KPIs.
    """
    raw = pd.read_csv(filepath, header=None, dtype=str).fillna('')
    section = None
    result = {'year': year}

    for _, row in raw.iterrows():
        vals = [v.strip() for v in row]

        # Find first non-empty cell (label) and next non-empty cell (value)
        label, value = '', ''
        for i, v in enumerate(vals):
            if v:
                label = v
                for j in range(i + 1, len(vals)):
                    if vals[j]:
                        value = vals[j]
                        break
                break

        if not label:
            continue

        # Update section state
        if label in SECTION_HEADERS:
            section = SECTION_HEADERS[label]
            continue

        # Reset section on unrelated headers
        if label in ('Network Size and Activity', 'Size of Network',
                     'Value of Services', 'Network Activity',
                     'Avg. Gross Margin per Boat', 'Avg. Gross Margin per Call'):
            section = 'network'
            continue

        # Service type rows — store under current section
        if section and label in COCKPIT_SERVICE_TYPES:
            col_name = f'{section}_{label.replace(" ", "_").replace("/", "_")}'
            result[col_name] = _to_float(value)
            continue

        # Network KPIs
        if 'unique boats' in label.lower():
            result['unique_boats'] = _to_float(value)
        elif 'total number of port calls' in label.lower():
            result['port_calls'] = _to_float(value)
        elif 'days spent in network' in label.lower():
            result['days_in_network'] = _to_float(value)

    return result

# Parse all years
cockpit_records = []
for year in range(2020, 2026):
    fp = DATA_DIR / f'cockpit_{year}.csv'
    if fp.exists():
        rec = parse_cockpit_file(fp, year)
        cockpit_records.append(rec)
        print(f'{year}: {len(rec)-1} KPIs extracted')
    else:
        print(f'{year}: file not found — skipped')

cockpit = pd.DataFrame(cockpit_records).set_index('year').sort_index()
print(f'\nShape: {cockpit.shape}')

2020: 35 KPIs extracted
2021: 35 KPIs extracted


2022: 35 KPIs extracted
2023: 35 KPIs extracted
2024: 35 KPIs extracted
2025: 35 KPIs extracted

Shape: (6, 35)


In [12]:
# Fix revenue_Total: 2024 and 2025 are clearly missing a decimal separator
# Compare sum of service types vs reported Total to detect and fix
rev_service_cols = [c for c in cockpit.columns
                    if c.startswith('revenue_') and c != 'revenue_Total']

print('=== REVENUE TOTAL CHECK ===')
for yr in cockpit.index:
    calc_total = cockpit.loc[yr, rev_service_cols].sum()
    reported   = cockpit.loc[yr, 'revenue_Total']
    ratio = reported / calc_total if calc_total > 0 else np.nan
    print(f'{yr}: sum of services = {calc_total:>12,.3f}  |  reported Total = {reported:>14,.3f}  |  ratio = {ratio:,.1f}')

    # If reported total is >100x the sum of parts, it's a decimal error
    if pd.notna(ratio) and ratio > 100:
        fixed = reported / 1000
        print(f'      → FIXING: {reported:,.0f} → {fixed:,.3f} (divided by 1000)')
        cockpit.loc[yr, 'revenue_Total'] = fixed

=== REVENUE TOTAL CHECK ===
2020: sum of services =       81.275  |  reported Total =         47.309  |  ratio = 0.6
2021: sum of services =      360.535  |  reported Total =        360.535  |  ratio = 1.0
2022: sum of services =      322.099  |  reported Total =        322.101  |  ratio = 1.0
2023: sum of services =      512.996  |  reported Total =        512.996  |  ratio = 1.0
2024: sum of services =    1,181.699  |  reported Total =  1,181,699.000  |  ratio = 1,000.0
      → FIXING: 1,181,699 → 1,181.699 (divided by 1000)
2025: sum of services =    1,541.398  |  reported Total =  1,541,399.000  |  ratio = 1,000.0
      → FIXING: 1,541,399 → 1,541.399 (divided by 1000)


In [13]:
# Same check for gm_Total
gm_service_cols = [c for c in cockpit.columns
                   if c.startswith('gm_') and c != 'gm_Total' and not c.startswith('gm_pct')]

print('=== GROSS MARGIN TOTAL CHECK ===')
for yr in cockpit.index:
    calc_total = cockpit.loc[yr, gm_service_cols].sum()
    reported   = cockpit.loc[yr, 'gm_Total']
    ratio = reported / calc_total if calc_total > 0 else np.nan
    print(f'{yr}: sum of services = {calc_total:>12,.3f}  |  reported Total = {reported:>14,.3f}  |  ratio = {ratio:,.1f}')

    if pd.notna(ratio) and ratio > 100:
        fixed = reported / 1000
        print(f'      → FIXING: {reported:,.0f} → {fixed:,.3f} (divided by 1000)')
        cockpit.loc[yr, 'gm_Total'] = fixed

print('\n=== NETWORK ACTIVITY ===')
net_cols = [c for c in ('unique_boats', 'port_calls', 'days_in_network') if c in cockpit.columns]
print(cockpit[net_cols])

=== GROSS MARGIN TOTAL CHECK ===
2020: sum of services =        8.974  |  reported Total =          5.977  |  ratio = 0.7
2021: sum of services =       93.838  |  reported Total =         93.838  |  ratio = 1.0
2022: sum of services =    1,135.599  |  reported Total =         56.679  |  ratio = 0.0
2023: sum of services =      133.147  |  reported Total =        133.147  |  ratio = 1.0
2024: sum of services =      340.773  |  reported Total =        340.773  |  ratio = 1.0
2025: sum of services =      360.591  |  reported Total =        360.589  |  ratio = 1.0

=== NETWORK ACTIVITY ===
      unique_boats  port_calls  days_in_network
year                                           
2020           1.0         0.0              NaN
2021          11.0        42.0            218.0
2022           7.0        10.0            147.0
2023          14.0        20.0            247.0
2024          26.0        43.0            804.0
2025          20.0        40.0            535.0


In [14]:
print('=== CLEANED COCKPIT ===')
cockpit

=== CLEANED COCKPIT ===


,revenue_Agency_Fee,revenue_Agency_Services,revenue_Bunkering,revenue_Hospitality,revenue_Port_Marina,revenue_Provisioning,revenue_Technical_Services,revenue_Total,gm_Agency_Fee,gm_Agency_Services,...,port_calls,days_in_network,network_Agency_Fee,network_Agency_Services,network_Bunkering,network_Hospitality,network_Port_Marina,network_Provisioning,network_Technical_Services,network_Total
year,,,,,,,,,,,,,,,,,,,,,
2020,1.844,34.000,0.000,45.431,0.000,0.000,0.000,47.309,1.844,3.000,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2021,87.733,44.237,3.357,13.866,181.403,6.764,23.175,360.535,87.733,6.105,...,42.0,218.0,2.089,145.0,0.000,0.0,0.000,0.000,0.0,2.234
2022,48.434,93.633,50.690,34.872,45.713,46.373,2.384,322.101,48.434,1.745,...,10.0,147.0,4.843,174.0,360.000,92.0,182.000,16.000,0.0,5.668
2023,87.716,24.666,174.811,73.984,62.866,82.934,6.019,512.996,74.194,5.286,...,20.0,247.0,3.710,264.0,1.179,242.0,418.000,786.000,58.0,6.657
2024,200.198,73.732,218.560,92.037,169.868,352.537,74.767,1181.699,192.254,19.480,...,43.0,804.0,4.471,453.0,846.000,335.0,173.000,1.356,292.0,7.925
2025,98.996,186.286,159.709,181.290,389.871,442.144,83.102,1541.399,98.082,35.373,...,40.0,535.0,2.452,884.0,767.000,675.0,1.359,2.362,515.0,9.015


---
## 4. Merge — Analysis-Ready Dataset

Join transactions with yacht specs to create a single analysis-ready table.
This enables direct correlation between cost/margin and yacht physical characteristics.

In [15]:
costs_merged = costs.merge(specs_join, on='yacht_id', how='left')

# Check join coverage
n_unmatched = costs_merged['gt'].isna().sum()
print(f'Merged rows: {len(costs_merged)}')
print(f'Unmatched (no specs): {n_unmatched}')
if n_unmatched > 0:
    print('Unmatched yacht IDs:', costs_merged[costs_merged['gt'].isna()]['yacht_id'].unique())

print('\nColumn list:')
print(costs_merged.columns.tolist())

print('\nSample rows:')
costs_merged[['yacht_id', 'year', 'service_type', 'service_category',
              'cost_no_vat', 'final_charge', 'margin', 'stay_days',
              'gt', 'loa_m', 'fuel_lph', 'size_category']].head(10)

Merged rows: 1654
Unmatched (no specs): 11
Unmatched yacht IDs: <StringArray>
['yacht_17']
Length: 1, dtype: str

Column list:
['yacht_id', 'office', 'arrival_port', 'arrival_date', 'departure_date', 'service_type', 'invoice_comments', 'supplier', 'cost_no_vat', 'final_charge', 'row_format', 'year', 'month', 'stay_days', 'margin', 'service_category', 'flag', 'gt', 'nt', 'loa_m', 'reg_length_m', 'beam_m', 'draft_m', 'air_draft_m', 'fuel_lph', 'loskrav', 'size_category']

Sample rows:


,yacht_id,year,service_type,service_category,cost_no_vat,final_charge,margin,stay_days,gt,loa_m,fuel_lph,size_category
0,yacht_10,2024,Repair of,Technical Services,335.68,457.98,122.30,8.0,497.0,53.25,130.0,Mellomstor
1,yacht_10,2024,Transfer Service,Hospitality,576.87,917.27,340.40,10.0,497.0,53.25,130.0,Mellomstor
2,yacht_10,2024,Purchasing Assistance,Agency Services,201.73,247.15,45.42,8.0,497.0,53.25,130.0,Mellomstor
3,yacht_10,2024,Custom Formalities,Agency Services,0.00,251.04,251.04,8.0,497.0,53.25,130.0,Mellomstor
4,yacht_10,2024,Purchasing Assistance,Agency Services,160.86,221.49,60.63,10.0,497.0,53.25,130.0,Mellomstor
5,yacht_10,2024,Provisioning,Provisioning,324.88,466.12,141.24,9.0,497.0,53.25,130.0,Mellomstor
6,yacht_10,2024,Port Dues,Port Marina,3665.83,4197.39,531.56,8.0,497.0,53.25,130.0,Mellomstor
7,yacht_10,2024,Mooring/Unmooring Assistance,Port Marina,115.75,140.80,25.05,8.0,497.0,53.25,130.0,Mellomstor
8,yacht_10,2024,Storage & Transport,Agency Services,135.69,186.45,50.76,9.0,497.0,53.25,130.0,Mellomstor
9,yacht_10,2024,Storage & Transport,Agency Services,140.57,159.41,18.84,8.0,497.0,53.25,130.0,Mellomstor


In [16]:
print('=== SUMMARY BY SIZE CATEGORY ===')
summary = (
    costs_merged
    .groupby('size_category')
    .agg(
        n_transactions=('final_charge', 'count'),
        n_yachts=('yacht_id', 'nunique'),
        total_revenue=('final_charge', 'sum'),
        total_cost=('cost_no_vat', 'sum'),
        total_margin=('margin', 'sum'),
        avg_margin_pct=('margin', lambda x: (x.sum() / costs_merged.loc[x.index, 'final_charge'].sum() * 100) if costs_merged.loc[x.index, 'final_charge'].sum() > 0 else np.nan),
    )
    .round(2)
)
print(summary)

print('\n=== SUMMARY BY SERVICE CATEGORY ===')
cat_summary = (
    costs_merged
    .groupby('service_category')
    .agg(
        n_transactions=('final_charge', 'count'),
        total_revenue=('final_charge', 'sum'),
        total_cost=('cost_no_vat', 'sum'),
        total_margin=('margin', 'sum'),
        avg_stay_days=('stay_days', 'mean'),
    )
    .sort_values('total_revenue', ascending=False)
    .round(2)
)
print(cat_summary)

=== SUMMARY BY SIZE CATEGORY ===
               n_transactions  n_yachts  total_revenue   total_cost  \
size_category                                                         
Liten                     566         5    10774894.85   8326542.46   
Mellomstor                469         8     7241844.42   5336281.59   
Stor                      617         4    22881452.23  18906543.81   

               total_margin  avg_margin_pct  
size_category                                
Liten            2448352.39           22.72  
Mellomstor       1905562.83           26.31  
Stor             3974908.42           17.37  

=== SUMMARY BY SERVICE CATEGORY ===
                    n_transactions  total_revenue   total_cost  total_margin  \
service_category                                                               
Port Marina                    386    11394766.98  10052591.07    1342175.91   
Provisioning                   150     8871353.22   7223796.97    1647556.25   
Agency Services         

---
## 5. Export Cleaned Files

In [17]:
# costs_clean: all transactions with service_category, drop internal row_format
costs_out = costs.drop(columns=['row_format'])
costs_out.to_csv(DATA_DIR / 'costs_clean.csv', index=False, encoding='utf-8')
print(f'Exported costs_clean.csv      — {len(costs_out)} rows, {len(costs_out.columns)} cols')

# specs_clean: keep full detail with all variants
specs_out = specs_clean.drop(columns=['Yacht'])
specs_out.to_csv(DATA_DIR / 'specs_clean.csv', index=False, encoding='utf-8')
print(f'Exported specs_clean.csv      — {len(specs_out)} rows')

# cockpit_clean: yearly KPIs with fixed totals
cockpit.to_csv(DATA_DIR / 'cockpit_clean.csv', encoding='utf-8')
print(f'Exported cockpit_clean.csv    — {len(cockpit)} rows, {len(cockpit.columns)} KPI cols')

# costs_merged: analysis-ready dataset
merged_out = costs_merged.drop(columns=['row_format'])
merged_out.to_csv(DATA_DIR / 'costs_merged.csv', index=False, encoding='utf-8')
print(f'Exported costs_merged.csv     — {len(merged_out)} rows, {len(merged_out.columns)} cols')

Exported costs_clean.csv      — 1654 rows, 16 cols

Exported specs_clean.csv      — 19 rows
Exported cockpit_clean.csv    — 6 rows, 35 KPI cols


Exported costs_merged.csv     — 1654 rows, 26 cols


---
## 6. Data Quality Summary

| File | Issue | Count | Action |
|------|-------|-------|--------|
| Kostnader_MM.csv | Colon typo i yacht ID (`Yacht:_2`, `Yacht:_14`) | 2 | Fikset under parsing |
| Kostnader_MM.csv | Subtotal-rader (ingen dato, bare beløp) | Mange | Hoppet over |
| Kostnader_MM.csv | `Not set` i VAT-kolonner | ~1 656 | Konvertert til NaN |
| Kostnader_MM.csv | Manglende/nullverdi i `final_charge` eller `cost_no_vat` | Se `flag`-kolonne | Flagget |
| Kostnader_MM.csv | 58 granulære tjenestetyper | Alle | Mappet til 7 cockpit-kategorier (`service_category`) |
| Kostnader_MM.csv | Rader utenfor 2020–2025 (f.eks. 2026) | 7 | Flagget (`out_of_year_range`) |
| Yacht-specs.csv | GT = 0 (yacht_17) | 1 | Satt til NaN |
| Yacht-specs.csv | Air Draft = 0 (yacht_1) | 1 | Satt til NaN |
| Yacht-specs.csv | NT mangler | yacht_7, yacht_17 | Beholdt som NaN |
| Yacht-specs.csv | Air Draft mangler | yacht_9, yacht_13, yacht_16, yacht_17 | Beholdt som NaN |
| Yacht-specs.csv | yacht_2 har 3 varianter | 3 rader | Aggregert til 1 rad for joining (`specs_join`) |
| cockpit_YYYY.csv | `revenue_Total` feilformatert (mangler desimaltegn) | 2024, 2025 | Delt på 1000 etter validering |
| cockpit_YYYY.csv | Avg. days per call/boat som Excel-seriedatoer | Alle år | Kolonnene hoppet over |
| cockpit_YYYY.csv | 2020-format har label i col[0] (ikke col[1]) | 2020 | Håndtert automatisk |